In [21]:
!pip install -q pymupdf sentence-transformers transformers sentencepiece accelerate pandas numpy

In [22]:
import requests
import os

pdf_url = "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf"
pdf_path = "/content/nist_ai_rmf.pdf"

response = requests.get(pdf_url, timeout=60)
response.raise_for_status()

with open(pdf_path, "wb") as f:
    f.write(response.content)

print("PDF downloaded successfully!")
print("File size:", round(os.path.getsize(pdf_path) / (1024 * 1024), 2), "MB")

PDF downloaded successfully!
File size: 1.86 MB


In [23]:
import fitz
import re

doc = fitz.open(pdf_path)

pages = []

for i, page in enumerate(doc):
    text = page.get_text("text")


    text = re.sub(r'(?<=\w)-\s*\n\s*(?=\w)', '', text)


    text = re.sub(r'\s+', ' ', text).strip()

    if len(text) > 50:
        pages.append({
            "page": i + 1,
            "text": text
        })

print("Total PDF pages:", len(doc))
print("Pages with usable text:", len(pages))
print("Total extracted words:", sum(len(p["text"].split()) for p in pages))

print("\nSample extracted text:")
print(pages[5]["text"][:1000])

Total PDF pages: 48
Pages with usable text: 48
Total extracted words: 15658

Sample extracted text:
NIST AI 100-1 AI RMF 1.0 Executive Summary Artificial intelligence (AI) technologies have significant potential to transform society and people’s lives – from commerce and health to transportation and cybersecurity to the environment and our planet. AI technologies can drive inclusive economic growth and support scientific advancements that improve the conditions of our world. AI technologies, however, also pose risks that can negatively impact individuals, groups, organizations, communities, society, the environment, and the planet. Like risks for other types of technology, AI risks can emerge in a variety of ways and can be characterized as long- or short-term, highor low-probability, systemic or localized, and high- or low-impact. The AI RMF refers to an AI system as an engineered or machine-based system that can, for a given set of objectives, generate outputs such as predictions, re

In [24]:
questions = [
    {"id": 1, "question": "How does the NIST AI RMF define risk?", "gold_pages": [9, 10]},
    {"id": 2, "question": "Why can third-party software, hardware, and data create challenges for AI risk measurement?", "gold_pages": [10, 11]},
    {"id": 3, "question": "What does risk tolerance mean in the AI RMF?", "gold_pages": [12]},
    {"id": 4, "question": "How should organizations prioritize AI risks, and what should they do when risks are unacceptable?", "gold_pages": [12, 13]},
    {"id": 5, "question": "Who are AI actors, and why should different perspectives be included in AI risk management?", "gold_pages": [15, 16]},
    {"id": 6, "question": "What characteristics does NIST identify as important for trustworthy AI systems?", "gold_pages": [17, 18, 19, 20, 21, 22, 23]},
    {"id": 7, "question": "What does the valid and reliable characteristic mean for an AI system?", "gold_pages": [17, 19]},
    {"id": 8, "question": "What does privacy-enhanced AI mean according to the framework?", "gold_pages": [17, 21]},
    {"id": 9, "question": "How does the framework address fairness and harmful bias in AI systems?", "gold_pages": [17, 23]},
    {"id": 10, "question": "What is the purpose of the GOVERN function in the NIST AI RMF?", "gold_pages": [27, 28, 29]},
    {"id": 11, "question": "What activities are covered by the MAP function?", "gold_pages": [30, 31, 32, 33]},
    {"id": 12, "question": "What is the purpose of the MEASURE function?", "gold_pages": [33, 34, 35, 36]},
    {"id": 13, "question": "What does the MANAGE function involve?", "gold_pages": [37, 38]},
    {"id": 14, "question": "What are AI RMF profiles, and why are they useful?", "gold_pages": [39, 40]},
    {"id": 15, "question": "How do AI risks differ from traditional software risks?", "gold_pages": [43, 44]}
]

print("Updated questions:", len(questions))

Updated questions: 15


In [25]:
page_lookup = {p["page"]: p["text"] for p in pages}

for item in questions:
    print("\n" + "=" * 80)
    print("Question", item["id"], ":", item["question"])
    print("Reference PDF pages:", item["gold_pages"])

    for page_no in item["gold_pages"]:
        if page_no in page_lookup:
            print(f"\n--- PDF page {page_no} ---")
            print(page_lookup[page_no][:450])
        else:
            print("WARNING: Page not found:", page_no)


Question 1 : How does the NIST AI RMF define risk?
Reference PDF pages: [9, 10]

--- PDF page 9 ---
NIST AI 100-1 AI RMF 1.0 Part 1: Foundational Information 1. Framing Risk AI risk management offers a path to minimize potential negative impacts of AI systems, such as threats to civil liberties and rights, while also providing opportunities to maximize positive impacts. Addressing, documenting, and managing AI risks and potential negative impacts effectively can lead to more trustworthy AI systems. 1.1 Understanding and Addressing Risks, Impact

--- PDF page 10 ---
NIST AI 100-1 AI RMF 1.0 Fig. 1. Examples of potential harms related to AI systems. Trustworthy AI systems and their responsible use can mitigate negative risks and contribute to benefits for people, organizations, and ecosystems. 1.2 Challenges for AI Risk Management Several challenges are described below. They should be taken into account when managing risks in pursuit of AI trustworthiness. 1.2.1 Risk Measurement AI risk

In [26]:
def create_chunks(pages, chunk_size, overlap):
    all_words = []

    for page in pages:
        for word in page["text"].split():
            all_words.append({
                "word": word,
                "page": page["page"]
            })

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(all_words), step):
        part = all_words[start:start + chunk_size]

        if not part:
            continue

        chunks.append({
            "chunk_id": len(chunks),
            "text": " ".join(x["word"] for x in part),
            "pages": sorted(set(x["page"] for x in part))
        })

        if start + chunk_size >= len(all_words):
            break

    return chunks


chunks_300 = create_chunks(pages, 300, 50)
chunks_600 = create_chunks(pages, 600, 100)

print("300-word chunks:", len(chunks_300))
print("600-word chunks:", len(chunks_600))
print("\nSample chunk:")
print(chunks_300[0]["text"][:700])
print("Source pages:", chunks_300[0]["pages"])


300-word chunks: 63
600-word chunks: 32

Sample chunk:
NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0) NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0) This publication is available free of charge from: https://doi.org/10.6028/NIST.AI.100-1 January 2023 U.S. Department of Commerce Gina M. Raimondo, Secretary National Institute of Standards and Technology Laurie E. Locascio, NIST Director and Under Secretary of Commerce for Standards and Technology Certain commercial entities, equipment, or materials may be identified in this document in order to describe an experimental procedure or concept adequately. Such identification is not intended to imply recommendation or endorsement by the Nation
Source pages: [1, 2, 3, 4]


In [27]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", device)

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

start_time = time.time()

texts_300 = [c["text"] for c in chunks_300]
texts_600 = [c["text"] for c in chunks_600]

embeddings_300 = embedding_model.encode(
    texts_300,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings_600 = embedding_model.encode(
    texts_600,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("\nEmbedding time:", round(time.time() - start_time, 2), "seconds")
print("300-word embedding shape:", embeddings_300.shape)
print("600-word embedding shape:", embeddings_600.shape)

Embedding device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding time: 25.21 seconds
300-word embedding shape: (63, 384)
600-word embedding shape: (32, 384)


In [28]:
def evaluate_retrieval(chunks, embeddings, questions):
    results = []

    question_texts = [q["question"] for q in questions]

    question_embeddings = embedding_model.encode(
        question_texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    similarities = np.dot(question_embeddings, embeddings.T)

    for i, question in enumerate(questions):
        top_indices = np.argsort(similarities[i])[::-1][:5]

        retrieved_pages = set()

        for idx in top_indices:
            retrieved_pages.update(chunks[idx]["pages"])

        gold_pages = set(question["gold_pages"])
        relevant_found = retrieved_pages.intersection(gold_pages)

        recall = len(relevant_found) / len(gold_pages)
        hit = 1 if len(relevant_found) > 0 else 0

        results.append({
            "question_id": question["id"],
            "question": question["question"],
            "gold_pages": sorted(gold_pages),
            "retrieved_pages": sorted(retrieved_pages),
            "recall_at_5": recall,
            "hit_at_5": hit,
            "top_chunk_ids": top_indices.tolist(),
            "top_similarities": similarities[i][top_indices].tolist()
        })

    return results


results_300 = evaluate_retrieval(chunks_300, embeddings_300, questions)
results_600 = evaluate_retrieval(chunks_600, embeddings_600, questions)

def print_metrics(results, label):
    avg_recall = np.mean([r["recall_at_5"] for r in results])
    hit_rate = np.mean([r["hit_at_5"] for r in results])

    print("\n", label)
    print("Average Recall@5:", round(avg_recall, 3))
    print("Hit@5:", round(hit_rate, 3))
    print("Successful hits:", sum(r["hit_at_5"] for r in results), "/", len(results))

print_metrics(results_300, "300 words / 50 overlap")
print_metrics(results_600, "600 words / 100 overlap")


 300 words / 50 overlap
Average Recall@5: 0.799
Hit@5: 0.933
Successful hits: 14 / 15

 600 words / 100 overlap
Average Recall@5: 0.881
Hit@5: 0.933
Successful hits: 14 / 15


In [29]:
import pandas as pd

comparison = pd.DataFrame({
    "Question ID": [q["id"] for q in questions],
    "Question": [q["question"] for q in questions],
    "Recall@5 (300)": [r["recall_at_5"] for r in results_300],
    "Recall@5 (600)": [r["recall_at_5"] for r in results_600],
    "Hit@5 (300)": [r["hit_at_5"] for r in results_300],
    "Hit@5 (600)": [r["hit_at_5"] for r in results_600]
})

display(comparison.round(3))

,Question ID,Question,Recall@5 (300),Recall@5 (600),Hit@5 (300),Hit@5 (600)
0,1,How does the NIST AI RMF define risk?,0.000,1.000,0,1
1,2,"Why can third-party software, hardware, and da...",1.000,1.000,1,1
2,3,What does risk tolerance mean in the AI RMF?,1.000,1.000,1,1
3,4,"How should organizations prioritize AI risks, ...",1.000,1.000,1,1
4,5,"Who are AI actors, and why should different pe...",1.000,1.000,1,1
5,6,What characteristics does NIST identify as imp...,0.571,0.714,1,1
6,7,What does the valid and reliable characteristi...,1.000,0.500,1,1
7,8,What does privacy-enhanced AI mean according t...,1.000,1.000,1,1
8,9,How does the framework address fairness and ha...,0.500,1.000,1,1
9,10,What is the purpose of the GOVERN function in ...,0.667,1.000,1,1


In [30]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

generator_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(generator_name)
generator = AutoModelForSeq2SeqLM.from_pretrained(generator_name)

generator_device = "cuda" if torch.cuda.is_available() else "cpu"
generator = generator.to(generator_device)

print("Generator loaded on:", generator_device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generator loaded on: cpu


In [31]:
selected_chunks = chunks_300
selected_embeddings = embeddings_300

question_texts = [q["question"] for q in questions]

question_embeddings = embedding_model.encode(
    question_texts,
    normalize_embeddings=True
)

similarities = np.dot(question_embeddings, selected_embeddings.T)

generation_results = []

for i, q in enumerate(questions):
    top_idx = int(np.argmax(similarities[i]))
    context = selected_chunks[top_idx]["text"][:1200]

    prompt = f"""
    Answer the question using only the context below.
    If the context does not contain the answer, say that the information
    is not available in the provided context.

    Context: {context}

    Question: {q["question"]}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(generator_device)

    with torch.no_grad():
        outputs = generator.generate(
            **inputs,
            max_new_tokens=100,
            num_beams=4,
            early_stopping=True
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    generation_results.append({
        "question_id": q["id"],
        "question": q["question"],
        "answer": answer,
        "retrieved_chunk_id": top_idx,
        "retrieved_pages": selected_chunks[top_idx]["pages"],
        "similarity": float(similarities[i][top_idx])
    })

    print("\nQuestion", q["id"], ":", q["question"])
    print("Retrieved pages:", selected_chunks[top_idx]["pages"])
    print("Answer:", answer)


Question 1 : How does the NIST AI RMF define risk?
Retrieved pages: [1, 2, 3, 4]
Answer: a living document

Question 2 : Why can third-party software, hardware, and data create challenges for AI risk measurement?
Retrieved pages: [10, 11]
Answer: They also may complicate risk measurement

Question 3 : What does risk tolerance mean in the AI RMF?
Retrieved pages: [11, 12]
Answer: readiness to bear the risk in order to achieve its objectives

Question 4 : How should organizations prioritize AI risks, and what should they do when risks are unacceptable?
Retrieved pages: [6, 7]
Answer: responsible development and use of AI systems

Question 5 : Who are AI actors, and why should different perspectives be included in AI risk management?
Retrieved pages: [15, 16, 17]
Answer: environmental groups, civil society organizations, end users, and potentially impacted individuals and communities

Question 6 : What characteristics does NIST identify as important for trustworthy AI systems?
Retrieved 

In [32]:
generation_df = pd.DataFrame(generation_results)

generation_df["quality"] = ""

display(generation_df[
    ["question_id", "question", "answer", "retrieved_pages", "quality"]
])

ratings = [""] * len(generation_df)

generation_df["quality"] = ratings

print("Answers ready for manual evaluation.")

,question_id,question,answer,retrieved_pages,quality
0,1,How does the NIST AI RMF define risk?,a living document,"[1, 2, 3, 4]",
1,2,"Why can third-party software, hardware, and da...",They also may complicate risk measurement,"[10, 11]",
2,3,What does risk tolerance mean in the AI RMF?,readiness to bear the risk in order to achieve...,"[11, 12]",
3,4,"How should organizations prioritize AI risks, ...",responsible development and use of AI systems,"[6, 7]",
4,5,"Who are AI actors, and why should different pe...","environmental groups, civil society organizati...","[15, 16, 17]",
5,6,What characteristics does NIST identify as imp...,"valid and reliable, safe, secure and resilient...","[16, 17]",
6,7,What does the valid and reliable characteristi...,necessary condition of trustworthiness,"[16, 17]",
7,8,What does privacy-enhanced AI mean according t...,"loss in accuracy, affect","[22, 23]",
8,9,How does the framework address fairness and ha...,NIST has identified three major categories of ...,"[22, 23, 24]",
9,10,What is the purpose of the GOVERN function in ...,"to govern, map, measure, and manage AI risks","[24, 25, 26]",


Answers ready for manual evaluation.


In [33]:
valid_ratings = ["Correct", "Partial", "Incorrect"]

if all(r in valid_ratings for r in ratings):
    print(generation_df["quality"].value_counts())
    print(
        "Strict correctness:",
        round((generation_df["quality"] == "Correct").mean() * 100, 2),
        "%"
    )
else:
    print("Please fill all 15 ratings with Correct, Partial, or Incorrect.")

Please fill all 15 ratings with Correct, Partial, or Incorrect.


In [34]:
comparison.to_csv(
    "/content/retrieval_comparison.csv",
    index=False
)

pd.DataFrame(results_300).to_csv(
    "/content/retrieval_results_300.csv",
    index=False
)

pd.DataFrame(results_600).to_csv(
    "/content/retrieval_results_600.csv",
    index=False
)

generation_df.to_csv(
    "/content/generation_results.csv",
    index=False
)


pd.DataFrame([
    {
        "chunk_id": c["chunk_id"],
        "text": c["text"],
        "pages": c["pages"]
    }
    for c in chunks_300
]).to_csv("/content/chunks_300.csv", index=False)

print("All result files exported successfully!")

All result files exported successfully!


In [35]:
from google.colab import files

for filename in [
    "retrieval_comparison.csv",
    "retrieval_results_300.csv",
    "retrieval_results_600.csv",
    "generation_results.csv",
    "chunks_300.csv"
]:
    files.download("/content/" + filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
ratings = [
    "Incorrect",
    "Partial",
    "Correct",
    "Incorrect",
    "Partial",
    "Partial",
    "Partial",
    "Incorrect",
    "Partial",
    "Partial",
    "Incorrect",
    "Correct",
    "Partial",
    "Partial",
    "Partial"
]

generation_df["quality"] = ratings

print("Answer quality distribution:")
print(generation_df["quality"].value_counts())

correct = (generation_df["quality"] == "Correct").sum()
partial = (generation_df["quality"] == "Partial").sum()
incorrect = (generation_df["quality"] == "Incorrect").sum()

print("\nFinal answer evaluation")
print("Correct:", correct)
print("Partial:", partial)
print("Incorrect:", incorrect)
print("Strict accuracy:", round(correct / len(ratings) * 100, 2), "%")

generation_df.to_csv(
    "/content/generation_results.csv",
    index=False
)

print("\nGeneration results saved.")

Answer quality distribution:
quality
Partial      9
Incorrect    4
Correct      2
Name: count, dtype: int64

Final answer evaluation
Correct: 2
Partial: 9
Incorrect: 4
Strict accuracy: 13.33 %

Generation results saved.


In [37]:
import json

with open("/content/evaluation_questions.json", "w") as f:
    json.dump(questions, f, indent=4)

summary = {
    "document": "NIST AI Risk Management Framework 1.0",
    "total_pdf_pages": len(doc),
    "usable_pages": len(pages),
    "total_words": sum(len(p["text"].split()) for p in pages),
    "chunks_300": len(chunks_300),
    "chunks_600": len(chunks_600),
    "embedding_time_seconds": 8.28,
    "recall_at_5_300": float(np.mean([r["recall_at_5"] for r in results_300])),
    "recall_at_5_600": float(np.mean([r["recall_at_5"] for r in results_600])),
    "hit_at_5_300": float(np.mean([r["hit_at_5"] for r in results_300])),
    "hit_at_5_600": float(np.mean([r["hit_at_5"] for r in results_600])),
    "answer_correct": int(correct),
    "answer_partial": int(partial),
    "answer_incorrect": int(incorrect)
}

with open("/content/task2_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

print(json.dumps(summary, indent=4))

{
    "document": "NIST AI Risk Management Framework 1.0",
    "total_pdf_pages": 48,
    "usable_pages": 48,
    "total_words": 15658,
    "chunks_300": 63,
    "chunks_600": 32,
    "embedding_time_seconds": 8.28,
    "recall_at_5_300": 0.7992063492063491,
    "recall_at_5_600": 0.880952380952381,
    "hit_at_5_300": 0.9333333333333333,
    "hit_at_5_600": 0.9333333333333333,
    "answer_correct": 2,
    "answer_partial": 9,
    "answer_incorrect": 4
}


In [38]:
from google.colab import files

final_files = [
    "retrieval_comparison.csv",
    "retrieval_results_300.csv",
    "retrieval_results_600.csv",
    "generation_results.csv",
    "chunks_300.csv",
    "evaluation_questions.json",
    "task2_summary.json"
]

for filename in final_files:
    files.download("/content/" + filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (4 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>